In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Annotated
import operator

class Task(BaseModel):
    name: str
    dependencies: list[str] = Field(default_factory=list)
    status: str = "PENDING"
# =========================================================
# 2. GLOBAL STATE
# =========================================================
class NodeState(BaseModel):

    tasks: list[Task] = Field(
        default_factory=list
    )

    completed_tasks: Annotated[
        list[str],
        operator.add
    ] = Field(
        default_factory=list
    )


# =========================================================
# 3. CREATE INITIAL TASKS
# =========================================================
def create_tasks(state: NodeState):
    tasks = [
        Task(
            name="Research",
            dependencies=[],
            status="PENDING"
        ),
        Task(
            name="FinanceReview",
            dependencies=["Research"],
            status="PENDING"
        ),
        Task(
            name="FinalReview",
            dependencies=["Research","FinanceReview"],
            status="PENDING"
        )
    ]
    print("\n========== TASKS CREATED ==========\n")
    for task in tasks:
        print(
            f"{task.name:15} | "
            f"Dependencies: {task.dependencies} | "
            f"Status: {task.status}"
        )
    return {
        "tasks": tasks
    }
# =========================================================
# 4. SCHEDULER
# =========================================================
def scheduler(state: NodeState):
    print("\n========== SCHEDULER ==========\n")
    ready_tasks = []
    for task in state.tasks:
        # ---------------------------------------------
        # Already completed task ko skip karo
        # ---------------------------------------------
        if task.name in state.completed_tasks:
            continue
        # ---------------------------------------------
        # Dependency check
        # ---------------------------------------------
        dependencies_satisfied = all(
            dependency in state.completed_tasks
            for dependency in task.dependencies
        )
        # ---------------------------------------------
        # READY
        # ---------------------------------------------

        if dependencies_satisfied:

            task.status = "READY"
            ready_tasks.append(task)

        # ---------------------------------------------
        # BLOCKED
        # ---------------------------------------------
        else:
            task.status = "BLOCKED"
    # ---------------------------------------------
    # Concurrency limit
    # ---------------------------------------------
    concurrency_limit = 2
    running_tasks = ready_tasks[:concurrency_limit]
    waiting_tasks = ready_tasks[concurrency_limit:]
    # ---------------------------------------------
    # RUNNING
    # ---------------------------------------------
    for task in running_tasks:
        task.status = "RUNNING"
    # ---------------------------------------------
    # WAITING
    # ---------------------------------------------
    for task in waiting_tasks:

        task.status = "WAITING"

    # ---------------------------------------------
    # Print scheduler state
    # ---------------------------------------------

    for task in state.tasks:

        print(
            f"{task.name} | "
            f"Dependencies: {task.dependencies} | "
            f"Status: {task.status}"
        )

    return {
        "tasks": state.tasks
    }


# =========================================================
# 5. RUNTIME
# =========================================================

def runtime(state: NodeState):

    print("\n========== RUNTIME ==========\n")

    completed_now = []

    for task in state.tasks:

        # Runtime sirf RUNNING tasks execute karega

        if task.status != "RUNNING":
            continue

        print(
            f"Executing: {task.name}"
        )
        # ---------------------------------------------
        # Simulated successful execution
        # ---------------------------------------------
        task.status = "SUCCESS"
        completed_now.append(task.name)
        print(
            f"{task.name} -> SUCCESS"
        )
    return {
        "tasks": state.tasks,
        "completed_tasks": completed_now
    }
# =========================================================
# 6. CHECK WHETHER MORE TASKS ARE AVAILABLE
# =========================================================
def check_remaining_tasks(state: NodeState):
    for task in state.tasks:
        if task.name not in state.completed_tasks:
            return "Scheduler"
    return "END"
# =========================================================
# 7. BUILD GRAPH
# =========================================================
graph = StateGraph(NodeState)
graph.add_node(
    "CreateTasks",
    create_tasks
)
graph.add_node(
    "Scheduler",
    scheduler
)
graph.add_node(
    "Runtime",
    runtime
)
# =========================================================
# 8. GRAPH FLOW
# =========================================================
graph.add_edge(
    START,
    "CreateTasks"
)
graph.add_edge(
    "CreateTasks",
    "Scheduler"
)
graph.add_edge(
    "Scheduler",
    "Runtime"
)
# =========================================================
# 9. CONDITIONAL ROUTING
# =========================================================
graph.add_conditional_edges(
    "Runtime",
    check_remaining_tasks,
    {
        "Scheduler": "Scheduler",
        "END": END
    }
)
# =========================================================
# 10. COMPILE
# =========================================================
app = graph.compile()
app
# =========================================================
# 11. EXECUTE
# =========================================================
result = app.invoke(
    {}
)
# =========================================================
# 12. FINAL STATE
# =========================================================
print("\n========== FINAL STATE ==========\n")
print(
    "Completed Tasks:",
    result["completed_tasks"]
)
print()
for task in result["tasks"]:
    print(
        f"Task: {task.name}"
    )
    print(
        f"Dependencies: {task.dependencies}"
    )

    print(
        f"Status: {task.status}"
    )

    print("-----------------------------------")

app


========== TASKS CREATED ==========

Research        | Dependencies: [] | Status: PENDING
FinanceReview   | Dependencies: ['Research', 'FinanceReview'] | Status: PENDING
Writer          | Dependencies: ['FinanceReview'] | Status: PENDING

========== SCHEDULER ==========

Research | Dependencies: [] | Status: RUNNING
FinanceReview | Dependencies: ['Research', 'FinanceReview'] | Status: BLOCKED
Writer | Dependencies: ['FinanceReview'] | Status: BLOCKED

========== RUNTIME ==========

Executing: Research
Research -> SUCCESS

========== SCHEDULER ==========

Research | Dependencies: [] | Status: SUCCESS
FinanceReview | Dependencies: ['Research', 'FinanceReview'] | Status: BLOCKED
Writer | Dependencies: ['FinanceReview'] | Status: BLOCKED

========== RUNTIME ==========


========== SCHEDULER ==========

Research | Dependencies: [] | Status: SUCCESS
FinanceReview | Dependencies: ['Research', 'FinanceReview'] | Status: BLOCKED
Writer | Dependencies: ['FinanceReview'] | Status: BLOCKED

=====

GraphRecursionError: Recursion limit of 10007 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT